<a href="https://colab.research.google.com/github/Fouzia13352/MLOPS1/blob/main/Preprocessing_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loading a Dataset into Colab

There are several ways to load datasets into your Colab environment:

1.  **Upload from your local machine:** You can use the file upload feature in the left sidebar (folder icon). After uploading, the file will be available in the Colab filesystem (e.g., `/content/your_file.csv`).
2.  **Mount Google Drive:** This is useful for larger datasets or if your data is already in Google Drive. You can mount your Google Drive directly.
3.  **From a URL:** If the dataset is hosted online (e.g., a CSV file on GitHub), you can download it directly using `!wget` or `pd.read_csv()` with the URL.
4.  **From Google Cloud Storage or BigQuery:** For larger-scale data, you can use client libraries like `google-cloud-storage` or `pandas-gbq`.

Below is an example of how to load a CSV file that has been uploaded to the Colab environment using `pandas`:

In [1]:
!pip install pandas

**Import Required Libraries**

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

**Load the Dataset**

In [4]:
data = pd.read_csv("/content/Titanic-Dataset.csv")

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


**Check missing values**

In [6]:
print(data.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**Select Relevant Features**

In [7]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

target = "Survived"

X = data[features]
y = data[target]

print(X.head())
print(y.head())

   Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0       3    male  22.0      1      0   7.2500        S
1       1  female  38.0      1      0  71.2833        C
2       3  female  26.0      0      0   7.9250        S
3       1  female  35.0      1      0  53.1000        S
4       3    male  35.0      0      0   8.0500        S
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


**Split the Dataset**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 712
Testing records: 179


**Before Proper Preprocessing**

In [10]:
numerical_only = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

baseline_train = X_train[numerical_only].copy()
baseline_test = X_test[numerical_only].copy()

train_valid_rows = baseline_train.dropna().index
test_valid_rows = baseline_test.dropna().index

baseline_train = baseline_train.loc[train_valid_rows]
baseline_test = baseline_test.loc[test_valid_rows]

y_train_baseline = y_train.loc[train_valid_rows]
y_test_baseline = y_test.loc[test_valid_rows]

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(baseline_train, y_train_baseline)

baseline_predictions = baseline_model.predict(baseline_test)

baseline_accuracy = accuracy_score(
    y_test_baseline,
    baseline_predictions
)

print("Baseline accuracy:", baseline_accuracy)
print("Training records used:", len(baseline_train))
print("Testing records used:", len(baseline_test))

Baseline accuracy: 0.6762589928057554
Training records used: 575
Testing records used: 139


**Define Numerical and Categorical Features**

In [11]:
numerical_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Sex",
    "Embarked"
]

**Numerical Preprocessing**
For numerical variables:

Replace missing values using the median.
Standardize the values.

In [12]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

**Categorical Preprocessing**

For categorical variables:

Replace missing values using the most frequent category.
Convert categories using one-hot encoding.

In [13]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

**Combine the Preprocessing Steps**

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

**Create the Complete Machine-Learning Pipeline**

In [15]:
complete_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [16]:
complete_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [17]:
processed_predictions = complete_pipeline.predict(X_test)

processed_accuracy = accuracy_score(
    y_test,
    processed_predictions
)

print("Accuracy after preprocessing:", processed_accuracy)

Accuracy after preprocessing: 0.8044692737430168


In [18]:
print(
    classification_report(
        y_test,
        processed_predictions
    )
)

              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



**Compare Before and After Preprocessing**

In [19]:
comparison = pd.DataFrame(
    {
        "Approach": [
            "Before proper preprocessing",
            "After preprocessing pipeline"
        ],
        "Accuracy": [
            baseline_accuracy,
            processed_accuracy
        ],
        "Training records used": [
            len(baseline_train),
            len(X_train)
        ],
        "Testing records used": [
            len(baseline_test),
            len(X_test)
        ]
    }
)

print(comparison)

                       Approach  Accuracy  Training records used  \
0   Before proper preprocessing  0.676259                    575   
1  After preprocessing pipeline  0.804469                    712   

   Testing records used  
0                   139  
1                   179  


**Generate the Processed Dataset**

In [20]:
X_train_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

In [21]:
feature_names = complete_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print(feature_names)

['numerical__Pclass' 'numerical__Age' 'numerical__SibSp'
 'numerical__Parch' 'numerical__Fare' 'categorical__Sex_female'
 'categorical__Sex_male' 'categorical__Embarked_C'
 'categorical__Embarked_Q' 'categorical__Embarked_S']


In [22]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [23]:
processed_train_df["Survived"] = y_train.values
processed_test_df["Survived"] = y_test.values

In [24]:
processed_train_df.head()

,numerical__Pclass,numerical__Age,numerical__SibSp,numerical__Parch,numerical__Fare,categorical__Sex_female,categorical__Sex_male,categorical__Embarked_C,categorical__Embarked_Q,categorical__Embarked_S,Survived
692,0.829568,-0.081135,-0.465084,-0.466183,0.513812,0.0,1.0,0.0,0.0,1.0,1
481,-0.370945,-0.081135,-0.465084,-0.466183,-0.662563,0.0,1.0,0.0,0.0,1.0,0
527,-1.571457,-0.081135,-0.465084,-0.466183,3.955399,0.0,1.0,0.0,0.0,1.0,0
855,0.829568,-0.887827,-0.465084,0.727782,-0.467874,1.0,0.0,0.0,0.0,1.0,1
801,-0.370945,0.110934,0.478335,0.727782,-0.115977,1.0,0.0,0.0,0.0,1.0,1


**Save the Processed Dataset**

In [25]:
processed_train_df.to_csv(
    "titanic_processed_train.csv",
    index=False
)

processed_test_df.to_csv(
    "titanic_processed_test.csv",
    index=False
)

In [26]:
from google.colab import files

files.download("titanic_processed_train.csv")
files.download("titanic_processed_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Save the Complete Pipeline**

In [27]:
joblib.dump(
    complete_pipeline,
    "titanic_preprocessing_model_pipeline.pkl"
)

['titanic_preprocessing_model_pipeline.pkl']

In [ ]:
files.download(
    "titanic_preprocessing_model_pipeline.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Reuse the Saved Pipeline**

In [28]:
loaded_pipeline = joblib.load(
    "titanic_preprocessing_model_pipeline.pkl"
)

In [30]:
new_passenger = pd.DataFrame(
    {
        "Pclass": [3],
        "Sex": ["female"],
        "Age": [25],
        "SibSp": [0],
        "Parch": [0],
        "Fare": [7.25],
        "Embarked": ["S"]
    }
)

In [31]:
prediction = loaded_pipeline.predict(new_passenger)

if prediction[0] == 1:
    print("Predicted result: Survived")
else:
    print("Predicted result: Did not survive")

Predicted result: Survived
